# Phase 2: Statistics & Hypothesis Testing

**Overarching question:** What factors are associated with inspection failures and higher violation counts among Chicago food establishments?

In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect('chicago_food_safety.db')

In [3]:
h1=pd.read_sql_query("""
SELECT Risk, Results
FROM Inspections
WHERE Results IN ('Pass', 'Fail', 'Pass w/ Conditions')
    AND "Risk" IN ('Risk 1 (High)', 'Risk 2 (Medium)', 'Risk 3 (Low)')
""",conn)
contingency_h1=pd.crosstab(h1['Risk'],h1['Results'])
contingency_h1

Results,Fail,Pass,Pass w/ Conditions
Risk,,,
Risk 1 (High),44764,124127,36306
Risk 2 (Medium),10602,28200,8303
Risk 3 (Low),5073,10370,2066


In [8]:
from scipy.stats import chi2_contingency

chi2_h1, p_h1, dof_h1, expected_h1 = chi2_contingency(contingency_h1)

print("Smallest expected count:", expected_h1.min())   
print(f"Chi-square = {chi2_h1:.2f}, p-value = {p_h1:.4g}")

Smallest expected count: 3028.9075500998847
Chi-square = 702.87, p-value = 8.353e-151


In [4]:
df_h2 = pd.read_sql_query("""
    SELECT i."Inspection ID", i."Facility Type",
           COUNT(v."Violations_code") AS violation_count
    FROM "Inspections" i
    LEFT JOIN "violations" v ON i."Inspection ID" = v."Inspection ID"
    WHERE i."Facility Type" IS NOT NULL
    GROUP BY i."Inspection ID", i."Facility Type"
""", conn)

top_facility_types = df_h2['Facility Type'].value_counts().head(6).index
df_h2_top = df_h2[df_h2['Facility Type'].isin(top_facility_types)]
df_h2_top.head()

,Inspection ID,Facility Type,violation_count
0,44247,Restaurant,6
1,44248,Restaurant,3
2,44249,Restaurant,1
5,44252,Restaurant,3
6,44253,Restaurant,4


In [9]:
from scipy.stats import f_oneway, kruskal

print(df_h2_top.groupby('Facility Type')['violation_count'].skew())  # >0 confirms right-skew

groups_h2 = [g['violation_count'].values for _, g in df_h2_top.groupby('Facility Type')]

f_stat, p_anova = f_oneway(*groups_h2)
h_stat, p_kw = kruskal(*groups_h2)

print(f"ANOVA:          F = {f_stat:.2f}, p = {p_anova:.4g}")
print(f"Kruskal-Wallis: H = {h_stat:.2f}, p = {p_kw:.4g}")

Facility Type
Bakery                             1.386217
Children's Services Facility       1.617279
Daycare Above and Under 2 Years    1.431922
Grocery Store                      1.358565
Restaurant                         1.477987
School                             1.173489
Name: violation_count, dtype: float64
ANOVA:          F = 372.27, p = 0
Kruskal-Wallis: H = 1140.43, p = 2.346e-244


In [5]:
df_h3 = pd.read_sql_query("""
    SELECT "Inspection Date", "Results"
    FROM "Inspections"
    WHERE "Results" IN ('Pass', 'Fail', 'Pass w/ Conditions')
""", conn, parse_dates=['Inspection Date'])

df_h3['season'] = df_h3['Inspection Date'].dt.month.isin([12, 1, 2]).map({True: 'Winter', False: 'Rest of year'})
df_h3['passed'] = df_h3['Results'].isin(['Pass', 'Pass w/ Conditions'])

summary_h3 = df_h3.groupby('season')['passed'].agg(['sum', 'count'])
summary_h3

,sum,count
season,,
Rest of year,162144,209561
Winter,47228,60250


In [10]:
from statsmodels.stats.proportion import proportions_ztest

print(summary_h3)   # check which row is which before reading the z below
counts = summary_h3['sum'].values
nobs = summary_h3['count'].values

z_stat, p_z = proportions_ztest(counts, nobs)
print(f"z = {z_stat:.2f}, p-value = {p_z:.4g}")

                 sum   count
season                      
Rest of year  162144  209561
Winter         47228   60250
z = -5.26, p-value = 1.45e-07


In [6]:
df_h4 = pd.read_sql_query("""
    SELECT i."Inspection ID", i."Inspection Date",
           COUNT(v."Violations_code") AS violation_count
    FROM "Inspections" i
    LEFT JOIN "violations" v ON i."Inspection ID" = v."Inspection ID"
    GROUP BY i."Inspection ID", i."Inspection Date"
""", conn, parse_dates=['Inspection Date'])

df_h4['year'] = df_h4['Inspection Date'].dt.year
df_h4[['year', 'violation_count']].head()

,year,violation_count
0,2010,6
1,2010,3
2,2010,1
3,2010,4
4,2010,4


In [11]:
from scipy.stats import pearsonr, spearmanr

r_pearson, p_pearson = pearsonr(df_h4['year'], df_h4['violation_count'])
r_spearman, p_spearman = spearmanr(df_h4['year'], df_h4['violation_count'])

print(f"Pearson  r = {r_pearson:.4f}, p = {p_pearson:.4g}")
print(f"Spearman r = {r_spearman:.4f}, p = {p_spearman:.4g}")

Pearson  r = 0.0229, p = 7.871e-38
Spearman r = -0.0197, p = 2.725e-28


In [7]:
conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_inspections_license_date_result
    ON "Inspections" ("License #", "Inspection Date", "Results")
""")
conn.commit()
df_h5 = pd.read_sql_query("""
    SELECT curr."Inspection ID",
           curr."Results" AS current_result,
           CASE WHEN EXISTS (
               SELECT 1 FROM "Inspections" prev
               WHERE prev."License #" = curr."License #"
                 AND prev."Inspection Date" < curr."Inspection Date"
                 AND prev."Results" = 'Fail'
           ) THEN 'Yes' ELSE 'No' END AS prior_fail
    FROM "Inspections" curr
    WHERE curr."Results" IN ('Pass', 'Fail', 'Pass w/ Conditions')
""", conn)

contingency_h5 = pd.crosstab(df_h5['prior_fail'], df_h5['current_result'])
contingency_h5

current_result,Fail,Pass,Pass w/ Conditions
prior_fail,,,
No,25813,62755,16548
Yes,34626,99942,30127


In [12]:
chi2_h5, p_h5, dof_h5, expected_h5 = chi2_contingency(contingency_h5)

print("Smallest expected count:", expected_h5.min())
print(f"Chi-square = {chi2_h5:.2f}, p-value = {p_h5:.4g}")

Smallest expected count: 18184.170771391826
Chi-square = 608.86, p-value = 6.145e-133
